# Week 1 · Day 3 — Lab 5
## Method Chaining, `.pipe()`, Pitfalls & the End-to-End Pipeline

> **AI Engineering Academy** · Gamut Technology Services · pandas 3.x

Everything from Days 1–3 comes together here. You'll write transformations as
**method chains** (no throwaway intermediate variables), inject custom, testable
steps with **`.pipe()`**, fix the **three pitfalls** every pandas engineer hits
(chained assignment, CSV intermediates, over-eager `apply`), and finish with a
**capstone**: a single-chain pipeline that ingests raw data, validates it, merges,
derives features, aggregates, and writes a typed Parquet artifact.

### Learning objectives
1. Refactor a multi-variable transformation into a single readable **method chain**.
2. Inject validation and custom logic mid-chain with **`.pipe()`**.
3. Recognize and fix the three pandas 3.x pitfalls with vectorized / `.loc` / Parquet solutions.
4. **Capstone:** build an end-to-end ingest→validate→merge→transform→aggregate→write pipeline as one chain.

### Time budget — ~75 min
| Segment | Time |
|---|---|
| Framing & objectives | 4 min |
| **A.** Method chaining | 14 min |
| **B.** `.pipe()` for custom steps | 14 min |
| **C.** The three pitfalls | 16 min |
| **CAP.** End-to-end pipeline | 25 min |
| Wrap-up | 2 min |

### Files you need (in `data/`)
- `users.csv` — 2000 users.
- `events.csv` — 8000 events.

*(The capstone writes a Parquet artifact into a local `artifacts/` folder.)*


In [ ]:
import pandas as pd
import numpy as np
import warnings, os
from pathlib import Path

print("pandas", pd.__version__)   # target: pandas 3.x on Python 3.13

# Solution is different here because of folder structure

DATA = Path("../data")
ART = Path("../artifacts")
ART.mkdir(exist_ok=True)

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

def _raises(fn):
    """True iff calling fn() raises an exception (used to test guard functions)."""
    try:
        fn()
        return False
    except Exception:
        return True

users = pd.read_csv(
    DATA / "users.csv",
    dtype={"user_id": "int32", "plan": "category", "region": "category"},
    parse_dates=["signup_date"],
)
events = pd.read_csv(
    DATA / "events.csv",
    dtype={"user_id": "int32", "model": "category", "score": "float32"},
    parse_dates=["event_date"],
)
print("users:", users.shape, "| events:", events.shape)

## Part A — Method chaining  *(guided)*

A chain expresses a transformation top-to-bottom with no intermediate variables to
fall out of sync. Each step returns a new object, so there's no mutation risk.
Read the chain as a recipe.


In [ ]:
# Without chaining: intermediate variables (df1, df2, ...) that can drift out of sync
d1 = events.dropna(subset=["score"])
d2 = d1[d1["score"] > 50]
d3 = d2.assign(tier=d2["score"] // 10 * 10)
result_unchained = d3.groupby("tier", observed=True)["score"].mean().reset_index()
print(result_unchained.head(3))

### Exercise A1 — Rewrite as one chain
Produce `chained` — the **same** result as `result_unchained` above — but as a
single method chain with no intermediate variables. Use `.dropna` → `.query` →
`.assign` → `.groupby` → `.mean` → `.reset_index`.


In [ ]:
chained = (
    events
    .dropna(subset=["score"])
    .query("score > 50")
    .assign(tier=lambda x: x["score"] // 10 * 10)
    .groupby("tier", observed=True)["score"]
    .mean()
    .reset_index()
)
print(chained.head(3))

In [ ]:
check("A1: chained matches the unchained result",
      lambda: chained.equals(result_unchained))
check("A1: chained is a DataFrame with a tier column",
      lambda: isinstance(chained, pd.DataFrame) and "tier" in chained.columns)

🧑‍🏫 **Instructor note — A1.** The win isn't fewer lines — it's fewer *names*.
Intermediate variables (`d1`, `d2`, `df_clean`) are a smell: they can be reused
after they're stale, and they invite mutation. A chain has exactly one output to
name. Wrap the chain in parentheses so each `.method` can start its own line.


## Part B — `.pipe()` for custom steps

`.pipe(fn, ...)` inserts your own function into a chain: it receives the DataFrame
and returns a DataFrame. This is how you put **validation** and **custom
transforms** inline — small, testable, side-effect-free functions.


In [ ]:
def add_normalized_score(df):
    lo, hi = df["score"].min(), df["score"].max()
    return df.assign(score_norm=(df["score"] - lo) / (hi - lo))

demo = events.pipe(add_normalized_score)
print(demo["score_norm"].agg(["min", "max"]).round(3))

### Exercise B1 — A validation pipe
Write `assert_no_nulls(df, cols)` that raises `ValueError` if any of `cols` contain
nulls, and otherwise returns `df` unchanged. Then build `validated` by piping
`events` through it on `cols=["user_id", "score"]` (which are clean, so it passes).


In [ ]:
def assert_no_nulls(df, cols):
    bad = df[cols].isna().sum()
    if bad.any():
        raise ValueError(f"Nulls found: {bad[bad > 0].to_dict()}")
    return df

validated = events.pipe(assert_no_nulls, cols=["user_id", "score"])
print("validation passed; rows:", len(validated))

In [ ]:
check("B1: validated passed through unchanged",
      lambda: validated is not None and len(validated) == len(events))
check("B1: assert_no_nulls RAISES on a column with nulls",
      lambda: _raises(lambda: events.assign(x=np.nan).pipe(assert_no_nulls, cols=["x"])))

🧑‍🏫 **Instructor note — B1.** The `_raises` helper (defined in the preamble) runs
the call and returns True only if it threw — so this check *confirms the guard
works* by feeding it a deliberately null column. Teach the pattern: validate at
input and output **boundaries**, not buried mid-chain. A pipe function that takes a
DataFrame and returns a DataFrame is trivial to unit-test with a 3-row fixture.


## Part C — The three pitfalls

The three mistakes engineers carry in from pandas 1.x — and their fixes.


### Pitfall 1 — Chained assignment
`df[mask][col] = value` edits a temporary copy: it warns and silently does nothing.
The fix is a single `.loc[mask, col] = value`.

**Exercise C1.** On `work = events.copy()`, set `score` to `0` for every
`atlas-mini` row **the correct way** with one `.loc`. Capture `n_zeroed`.


In [ ]:
work = events.copy()
work.loc[work["model"] == "atlas-mini", "score"] = 0
n_zeroed = int((work["score"] == 0).sum())
print("rows zeroed:", n_zeroed)

In [ ]:
check("C1: exactly the atlas-mini rows were zeroed",
      lambda: n_zeroed == int((events["model"] == "atlas-mini").sum()))
check("C1: non-atlas-mini scores untouched",
      lambda: work.loc[work["model"] != "atlas-mini", "score"].equals(
              events.loc[events["model"] != "atlas-mini", "score"]))

### Pitfall 3 — Reaching for `apply` too quickly
A row-wise `apply` is a Python loop. Almost always there's a vectorized equivalent
that returns the identical result far faster.

**Exercise C2.** Produce `label` two ways and confirm they match: `label_apply`
via `events["score"].apply(lambda s: "pass" if s >= 60 else "fail")`, and
`label_vec` via `np.where(events["score"] >= 60, "pass", "fail")`.


In [ ]:
label_apply = events["score"].apply(lambda s: "pass" if s >= 60 else "fail")
label_vec = np.where(events["score"] >= 60, "pass", "fail")
same = np.array_equal(label_apply.to_numpy(), label_vec)
print("identical result:", same)
print(pd.Series(label_vec).value_counts().to_dict())

In [ ]:
check("C2: vectorized result equals the apply result", lambda: same is True)

🧑‍🏫 **Instructor note — C.** Pitfall 1 (chained assignment) is *the* migration
failure — the silent no-op. Pitfall 3: `apply` returns the right answer, so it's
seductive; the cost is speed (a Python loop vs a C loop). The message isn't "never
`apply`" — it's "search for `np.where` / `.str` / `.dt` / `pd.cut` first, and only
`apply` when the logic is genuinely complex and the frame is small." (Pitfall 2 —
CSV for intermediates — was proven in Lab 2: use Parquet.)


## CAPSTONE — End-to-end pipeline  *(~25 min)*

Build **one method chain** that turns the two raw CSVs into a clean, typed,
aggregated Parquet artifact — no intermediate variables, no mutation. This mirrors a
real ingestion job:

1. **Merge** `users` → `events` (validated `1:m`).
2. **Validate** with a `.pipe` (no nulls in the key columns).
3. **Filter** to real scored events (`score > 0`).
4. **Derive** `days_active` and a `tier` band with `assign` + `pd.cut`.
5. **Aggregate** mean score by `plan` × `tier`.
6. **Write** the result to Parquet (PyArrow, snappy).


### CAP-1 — Build the pipeline chain
Fill in the chain to produce `summary` (a DataFrame with columns `plan`, `tier`,
`score`). Then write it to `artifacts/summary.parquet`. Use `observed=True` on the
groupby.


In [ ]:
summary = (
    users
    .merge(events, on="user_id", how="left", validate="1:m")
    .pipe(assert_no_nulls, cols=["user_id"])
    .query("score > 0")
    .assign(
        days_active=lambda x: (x["event_date"] - x["signup_date"]).dt.days,
        tier=lambda x: pd.cut(x["score"], bins=[0, 60, 80, 100],
                              labels=["low", "mid", "high"]),
    )
    .groupby(["plan", "tier"], observed=True)["score"]
    .mean()
    .reset_index()
)
summary.to_parquet(ART / "summary.parquet", engine="pyarrow", compression="snappy")
print(summary)

In [ ]:
check("CAP-1: summary has plan, tier, score columns",
      lambda: set(summary.columns) == {"plan", "tier", "score"})
check("CAP-1: one row per plan x tier that occurs",
      lambda: len(summary) == summary[["plan", "tier"]].drop_duplicates().shape[0])
check("CAP-1: null plans were dropped by groupby",
      lambda: not summary["plan"].isna().any())
check("CAP-1: mean scores are in valid range",
      lambda: bool(((summary["score"] >= 0) & (summary["score"] <= 100)).all()))
check("CAP-1: parquet artifact was written",
      lambda: (ART / "summary.parquet").exists())

### CAP-2 — Verify the artifact round-trips
Read `artifacts/summary.parquet` back into `reloaded` and confirm it equals
`summary` (dtypes and values intact). This is the type-safety payoff of Parquet.


In [ ]:
reloaded = pd.read_parquet(ART / "summary.parquet", engine="pyarrow")
matches = reloaded.equals(summary)
print("round-trip identical:", matches)
print("reloaded dtypes:\n", reloaded.dtypes)

In [ ]:
check("CAP-2: reloaded equals summary (dtypes + values)",
      lambda: matches is True)
check("CAP-2: tier survived as categorical",
      lambda: str(reloaded["tier"].dtype) == "category")

🧑‍🏫 **Instructor note — CAP.** This is the whole day in one chain: typed ingestion
(Lab 2), a validated merge and derived features (Lab 4), readable chaining and a
validation pipe (Lab 5), written to a typed Parquet artifact. Point out two design
wins: `query("score > 0")` doubles as the "drop users with no events" step (their
NaN scores fail the filter), and `groupby` silently drops the 20 null-plan rows so
the summary is clean. `pd.cut` produces a *categorical* `tier`, and Parquet brings
it back as `category` — CSV never could.


## Wrap-up — what you can now do

- Refactor multi-step transformations into a single, mutation-free **method chain**.
- Inject validation and custom logic with **`.pipe()`** — small, testable functions.
- Fix the three pandas 3.x pitfalls: `.loc[mask, col]` for updates, Parquet for intermediates, vectorized ops over `apply`.
- Build a complete **ingest → validate → merge → transform → aggregate → write** pipeline end-to-end.

**That's Day 3.** You can construct and reason about Series/DataFrame/Index, do
typed I/O across CSV/Parquet/JSON, inspect and select precisely, transform with
vectorized ops and groupby/merge/reshape, and compose it all into clean,
production-shaped pipelines.
